# **Traffic analysis on one demo video**

## Objectives
1. **Car Counting**: Count vehicles by type (car, bus, van) without double counting
2. **Congestion Detection**: Detect congestion level (sparse, light, medium, heavy)

## Source Data
- **Trained YOLO Model**: `results/weights/best.pt`
- **Test Image**: `data_processed/test/demo_video`
- **Detected Classes**: 0=car, 1=bus, 2=van

# 1. Libraries import

In [17]:
import os
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict, deque
import json
from datetime import datetime
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# YOLO
try:
    from ultralytics import YOLO
    print("YOLO available")
except ImportError:
    print("YOLO not installed. Installing...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'ultralytics'])
    from ultralytics import YOLO
    print("YOLO installed")

YOLO available


## 2. Path Configuration

In [18]:
# Paths (notebook is in notebooks/ folder)
PROJECT_ROOT = Path.cwd().parent

# Model path (from Training notebook)
MODEL_PATH = PROJECT_ROOT / 'results' / 'weights' / 'best.pt'

# Video source
DEMO_DIR = PROJECT_ROOT / 'data_processed' / 'test' / 'demo_video.mp4'

# Output directory
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'traffic_analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Class mapping (from training)
CLASS_MAPPING = {0: 'car', 1: 'bus', 2: 'van'}

print(f"Project root: {PROJECT_ROOT}")
print(f"Model path: {MODEL_PATH}")
print(f"Video directory: {DEMO_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"\nConfiguration complete")

Project root: c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project
Model path: c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\results\weights\best.pt
Video directory: c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\data_processed\test\demo_video.mp4
Output directory: c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\results\traffic_analysis

Configuration complete


## 3. CentroidTracker class for vehicle tracking

Tracking algorithm based on centroids to avoid counting the same vehicle multiple times.

In [19]:
class CentroidTracker:
    """
    Tracker based on centroids.
    Associates detections between frames to avoid double counting.
    """
    def __init__(self, maxDisappeared=30, maxDistance=50):
        """
        Args:
            maxDisappeared: Number of frames before considering a vehicle disappeared
            maxDistance: Maximum distance (pixels) to associate a detection with an ID
        """
        self.nextObjectID = 0
        self.objects = {}  # {ID: centroid}
        self.disappeared = {}  # {ID: number of disappeared frames}
        self.vehicle_types = {}  # {ID: class_name}
        self.vehicle_history = defaultdict(list)  # {ID: [(frame, bbox, class), ...]}
        self.maxDisappeared = maxDisappeared
        self.maxDistance = maxDistance

    def register(self, centroid, class_name):
        """Register a new object"""
        self.objects[self.nextObjectID] = centroid
        self.disappeared[self.nextObjectID] = 0
        self.vehicle_types[self.nextObjectID] = class_name
        self.nextObjectID += 1

    def deregister(self, objectID):
        """Deregister an object"""
        del self.objects[objectID]
        del self.disappeared[objectID]
        del self.vehicle_types[objectID]

    def update(self, rects, class_names, frame_num):
        """
        Update with new detections.
        
        Args:
            rects: List of [(x1, y1, x2, y2), ...]
            class_names: List of classes
            frame_num: Frame number
        
        Returns:
            Dict {ID: centroid}
        """
        if len(rects) == 0:
            # No detections - increment disappeared counters
            for objectID in list(self.disappeared.keys()):
                self.disappeared[objectID] += 1
                if self.disappeared[objectID] > self.maxDisappeared:
                    self.deregister(objectID)
            return self.objects

        # Calculate centroids
        input_centroids = np.zeros((len(rects), 2))
        for i, (x1, y1, x2, y2) in enumerate(rects):
            cX = (x1 + x2) // 2
            cY = (y1 + y2) // 2
            input_centroids[i] = [cX, cY]

        if len(self.objects) == 0:
            # First frame - register all detections
            for i in range(len(input_centroids)):
                self.register(input_centroids[i], class_names[i])
                self.vehicle_history[self.nextObjectID - 1].append({
                    'frame': frame_num,
                    'bbox': rects[i],
                    'class': class_names[i]
                })
        else:
            # Associate detections with existing objects
            objectIDs = list(self.objects.keys())
            objectCentroids = list(self.objects.values())

            # Calculate distances
            D = np.zeros((len(objectCentroids), len(input_centroids)))
            for i in range(len(objectCentroids)):
                for j in range(len(input_centroids)):
                    D[i, j] = np.linalg.norm(objectCentroids[i] - input_centroids[j])

            # Greedy association
            rows = D.min(axis=1).argsort()
            cols = D.argmin(axis=1)[rows]

            used_rows = set()
            used_cols = set()

            for row, col in zip(rows, cols):
                if row in used_rows or col in used_cols:
                    continue
                if D[row, col] > self.maxDistance:
                    continue

                objectID = objectIDs[row]
                self.objects[objectID] = input_centroids[col]
                self.disappeared[objectID] = 0
                self.vehicle_types[objectID] = class_names[col]
                
                self.vehicle_history[objectID].append({
                    'frame': frame_num,
                    'bbox': rects[col],
                    'class': class_names[col]
                })

                used_rows.add(row)
                used_cols.add(col)

            # Non-matched rows and columns
            unused_rows = set(range(D.shape[0])).difference(used_rows)
            unused_cols = set(range(D.shape[1])).difference(used_cols)

            if D.shape[0] >= D.shape[1]:
                for row in unused_rows:
                    objectID = objectIDs[row]
                    self.disappeared[objectID] += 1
                    if self.disappeared[objectID] > self.maxDisappeared:
                        self.deregister(objectID)
            else:
                for col in unused_cols:
                    self.register(input_centroids[col], class_names[col])
                    self.vehicle_history[self.nextObjectID - 1].append({
                        'frame': frame_num,
                        'bbox': rects[col],
                        'class': class_names[col]
                    })

        return self.objects

    def get_statistics(self):
        """Return counting statistics"""
        vehicle_counts = {'car': 0, 'bus': 0, 'van': 0}
        for vehicle_id, history in self.vehicle_history.items():
            if history:
                vehicle_class = history[0]['class']
                if vehicle_class in vehicle_counts:
                    vehicle_counts[vehicle_class] += 1
        return vehicle_counts

print("CentroidTracker class defined")

CentroidTracker class defined


## 4. CongestionDetector class for detecting traffic jams

Analyses vehicle density and speed to detect congestion.

In [20]:
# ======================
# SPEED PARAMETERS
# ======================
SMOOTH_N = 10   # sliding window size for smoothing
LOWER_ZONE_RATIO = 0.5  # only compute speed in the lower 50% of the frame
# This value comes from the calibration output in notebook 5
METER_PER_PIXEL = 0.02449574706793445

In [21]:
class CongestionDetector:
    """
    Detect the congestion level based on:
    - Vehicle density (vehicles/pixel)
    - Average speed (pixels/frame)
    - Average flow (vehicles passing per frame)
    """
    def __init__(self, window_size=30, fps=25, road_length_px=1500):
        """
        Args:
            window_size: Number of frames for moving average calculation
            fps: Frames per second (to convert flow)
            road_length_px: Length of the analysis area in pixels
        """
        self.window_size = window_size
        self.fps = fps
        self.road_length_px = road_length_px
        
        self.density_history = deque(maxlen=window_size)  # vehicles/pixel
        self.speed_history = deque(maxlen=window_size)    # pixels/frame
        self.flow_history = deque(maxlen=window_size)     # vehicles/frame
        self.congestion_levels = []
        self.total_vehicles_counted = 0  # Total count of vehicles passing
        
        # Density thresholds (vehicles/pixel) ~ old thresholds veh/m / 20 px/m
        self.DENSITY_SPARSE = 0.0025
        self.DENSITY_LIGHT = 0.0075
        self.DENSITY_MEDIUM = 0.015
        # Above = HEAVY
        
        # Flow thresholds (vehicles per second)
        self.FLOW_SPARSE = 0.2
        self.FLOW_LIGHT = 0.6
        self.FLOW_MEDIUM = 1.2
        
        # Speed thresholds (pixels/frame) ~ conversion of old km/h thresholds
        self.SPEED_STOPPED_PX = 0.05
        self.SPEED_SLOW_PX = 3.3
        self.SPEED_NORMAL_PX = 11.0
    
    def convert_speed_to_kmh(self, speed_pixels_per_frame):
        """Identity: returns speed in pixels/frame (no conversion)."""
        return speed_pixels_per_frame
    
    def convert_density_to_per_pixel(self, num_vehicles):
        """Convert vehicle density/frame to vehicles/pixel."""
        if self.road_length_px <= 0:
            return 0
        return num_vehicles / self.road_length_px
    
    def update(self, num_vehicles, avg_speed_pixels_per_frame, num_vehicles_crossed=0):
        """Update with current frame data."""
        density_per_pixel = self.convert_density_to_per_pixel(num_vehicles)
        self.density_history.append(density_per_pixel)
        
        speed_px = avg_speed_pixels_per_frame
        self.speed_history.append(speed_px)
        
        # Flow: number of vehicles crossing per frame
        self.flow_history.append(num_vehicles_crossed)
        self.total_vehicles_counted += num_vehicles_crossed
    
    def get_average_flow_rate(self):
        """Calculate average flow rate."""
        if not self.flow_history:
            return {
                'flow_per_frame': 0,
                'flow_per_second': 0,
                'flow_per_minute': 0
            }
        
        avg_flow_per_frame = np.mean(self.flow_history)
        avg_flow_per_second = avg_flow_per_frame * self.fps
        avg_flow_per_minute = avg_flow_per_second * 60
        
        return {
            'flow_per_frame': avg_flow_per_frame,
            'flow_per_second': avg_flow_per_second,
            'flow_per_minute': avg_flow_per_minute
        }
    
    def get_congestion_level(self):
        """Calculate the current congestion level."""
        if not self.density_history:
            return 'sparse'
        
        avg_density = np.mean(self.density_history)      # veh/pixel
        avg_speed_px = np.mean(self.speed_history) if self.speed_history else 0  # px/frame
        
        # Logic combining density and speed (units px)
        if avg_density < self.DENSITY_SPARSE:
            level = 'sparse'
        elif avg_density < self.DENSITY_LIGHT:
            level = 'light'
        elif avg_density < self.DENSITY_MEDIUM:
            if avg_speed_px < self.SPEED_SLOW_PX:
                level = 'heavy'
            else:
                level = 'medium'
        else:
            if avg_speed_px < self.SPEED_STOPPED_PX:
                level = 'heavy'
            else:
                level = 'heavy'
        
        return level
    
    def get_statistics(self):
        """Return complete congestion statistics (px/frame, veh/px)."""
        if not self.density_history:
            return {
                'avg_density_veh_per_px': 0,
                'max_density_veh_per_px': 0,
                'avg_speed_px_per_frame': 0,
                'max_speed_px_per_frame': 0,
                'avg_flow_per_frame': 0,
                'avg_flow_per_second': 0,
                'avg_flow_per_minute': 0,
                'total_vehicles_counted': 0,
                'congestion_level': 'sparse'
            }
        
        flow_rate = self.get_average_flow_rate()
        
        return {
            'avg_density_veh_per_px': np.mean(self.density_history),
            'max_density_veh_per_px': np.max(self.density_history),
            'avg_speed_px_per_frame': np.mean(self.speed_history) if self.speed_history else 0,
            'max_speed_px_per_frame': np.max(self.speed_history) if self.speed_history else 0,
            'avg_flow_per_frame': flow_rate['flow_per_frame'],
            'avg_flow_per_second': flow_rate['flow_per_second'],
            'avg_flow_per_minute': flow_rate['flow_per_minute'],
            'total_vehicles_counted': self.total_vehicles_counted,
            'congestion_level': self.get_congestion_level()
        }

print("CongestionDetector class defined")

CongestionDetector class defined


In [22]:
def speed_px_s(track, fps):  
    """
    Calculates speed in pixels per second.
    track: list of (frame_id, cx, cy)
    """
    if len(track) < SMOOTH_N + 1:
        return None

    # Get positions from SMOOTH_N frames ago and current frame
    f1, x1, y1 = track[-(SMOOTH_N + 1)]
    f2, x2, y2 = track[-1]

    dt = (f2 - f1) / fps
    if dt <= 0:
        return None

    # distance / time
    return np.hypot(x2 - x1, y2 - y1) / dt

## 5. Main video processing function

In [27]:
def process_video_with_counting_and_congestion(
    video_path,
    model,
    conf_threshold=0.5,
    output_video_path=None,
    show_progress=True
):
    """
    Process a video: detection, tracking (ID), real speed (km/h), flow, density, and congestion.
    """
    SMOOTH_N = 10   # Sliding window size for smoothing
    LOWER_ZONE_RATIO = 0.3  # Focus on the lower part of the frame
    METER_PER_PIXEL = 0.02449574706793445  # Calibration result

    # 1. Open the video
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Unable to open: {video_path}")
        return None
    
    # 2. Properties and initialization
    fps = cap.get(cv2.CAP_PROP_FPS) # Use float for precise math
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = total_frames / fps if fps > 0 else 0
    
    tracker = CentroidTracker(maxDisappeared=30, maxDistance=50)
    congestion = CongestionDetector(window_size=30)
    
    # New history for speed calculation
    speed_history = defaultdict(list) 
    
    out = None
    if output_video_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(str(output_video_path), fourcc, fps, (width, height))
    
    frame_num = 0
    congestion_history = []
    pbar = tqdm(total=total_frames, desc="Analysis in progress") if show_progress else None
    
    congestion_colors = {
        'sparse': (0, 255, 0), 'light': (0, 255, 255),
        'medium': (0, 165, 255), 'heavy': (0, 0, 255)
    }

    while True:
        ret, frame = cap.read()
        if not ret: break
        
        frame_num += 1
        H, W = frame.shape[:2]
        Y_MIN = H * (1 - LOWER_ZONE_RATIO) # Calibration zone boundary

        results = model(frame, conf=conf_threshold, verbose=False)
        rects = []
        class_names = []
        
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                cls_id = int(box.cls[0])
                if cls_id in CLASS_MAPPING:
                    rects.append((x1, y1, x2, y2))
                    class_names.append(CLASS_MAPPING[cls_id])
        
        tracked_objects = tracker.update(rects, class_names, frame_num)
        
        # 3. ADAPTED SPEED CALCULATION (km/h)
        current_frame_speeds_kmh = []
        for obj_id, centroid in tracked_objects.items():
            cx, cy = centroid
            
            # Only compute speed if vehicle is in the lower zone
            if cy >= Y_MIN:
                speed_history[obj_id].append((frame_num, cx, cy))
                
                # Check if we have enough points for a smoothed window
                if len(speed_history[obj_id]) >= SMOOTH_N + 1:
                    f1, x1, y1 = speed_history[obj_id][-(SMOOTH_N + 1)]
                    f2, x2, y2 = speed_history[obj_id][-1]
                    
                    dt = (f2 - f1) / fps
                    if dt > 0:
                        dist_px = np.hypot(x2 - x1, y2 - y1)
                        v_px_s = dist_px / dt
                        # Convert to km/h
                        v_kmh = v_px_s * METER_PER_PIXEL * 3.6
                        current_frame_speeds_kmh.append(v_kmh)
        
        avg_speed_kmh = np.mean(current_frame_speeds_kmh) if current_frame_speeds_kmh else 0
        
        # 4. FLOW & CONGESTION
        elapsed_minutes = frame_num / (fps * 60) if fps > 0 else 0.001
        total_unique_vehicles = len(tracker.vehicle_history)
        current_flow = total_unique_vehicles / elapsed_minutes
        
        # Use km/h for congestion analysis for better accuracy
        congestion.update(len(tracked_objects), avg_speed_kmh)
        current_level = congestion.get_congestion_level()
        current_density = congestion.convert_density_to_per_pixel(len(tracked_objects))
        
        if output_video_path:
            annotated = frame.copy()
            # Draw calibration line
            cv2.line(annotated, (0, int(Y_MIN)), (W, int(Y_MIN)), (255, 0, 0), 2)
            
            for obj_id, centroid in tracked_objects.items():
                if obj_id in tracker.vehicle_history and tracker.vehicle_history[obj_id]:
                    last_entry = tracker.vehicle_history[obj_id][-1]
                    x1, y1, x2, y2 = last_entry['bbox']
                    v_class = last_entry['class']
                    
                    color = (0, 255, 0) if v_class == 'car' else (0, 165, 255)
                    cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
                    
                    # Display Speed in KM/H on Label
                    v_text = ""
                    # Try to find specific vehicle speed
                    if len(speed_history[obj_id]) >= SMOOTH_N + 1:
                        # Re-calculate or retrieve for label
                        v_text = f" {current_frame_speeds_kmh[-1]:.1f}km/h" if current_frame_speeds_kmh else ""
                    
                    label = f"ID:{obj_id} {v_class}{v_text}"
                    cv2.putText(annotated, label, (x1, y1-5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            
            # Info overlay with density
            info_overlay = [
                f"Status: {current_level.upper()}",
                f"Avg Speed: {avg_speed_kmh:.1f} km/h",
                f"Density: {current_density:.5f} veh/px",
                f"Flow: {current_flow:.1f} veh/min",
                f"Total: {total_unique_vehicles}"
            ]
            
            overlay_color = congestion_colors[current_level]
            for i, text in enumerate(info_overlay):
                cv2.putText(annotated, text, (15, 40 + i*35),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,0), 4)
                cv2.putText(annotated, text, (15, 40 + i*35),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, overlay_color, 2)
            
            out.write(annotated)

        # Update History
        congestion_history.append({
            'frame': frame_num,
            'num_vehicles': len(tracked_objects),
            'avg_speed_kmh': avg_speed_kmh,
            'density_veh_per_px': current_density,
            'congestion': current_level
        })
        
        if pbar: pbar.update(1)

    if pbar: pbar.close()
    cap.release()
    if out: out.release()

    return {
        'total_vehicles': total_unique_vehicles,
        'avg_flow_rate_min': round(total_unique_vehicles / (duration_sec / 60), 2),
        'congestion_stats': congestion.get_statistics(),
        'congestion_history': congestion_history
    }

## 6. Demo Video Processing

In [28]:
# Loading the model
if MODEL_PATH.exists():
    print(f"Loading model: {MODEL_PATH}")
    model = YOLO(str(MODEL_PATH))
else:
    print("Custom model not found. Using yolov8n.pt")
    model = YOLO("yolov8n.pt")

# Use the video defined in DEMO_DIR
demo_video_path = DEMO_DIR

if demo_video_path.exists():
    print(f"Video found: {demo_video_path.name}")
else:
    print(f"Video not found: {demo_video_path}")
    print("Check the DEMO_DIR path in the configuration cell")

# Define the output path
output_video_path = OUTPUT_DIR / f"analyzed_{demo_video_path.name if demo_video_path.exists() else 'demo.mp4'}"

# Process the video
if demo_video_path.exists():
    print(f"\nProcessing video...")
    results = process_video_with_counting_and_congestion(
        video_path=demo_video_path,
        model=model,
        conf_threshold=0.25,
        output_video_path=output_video_path,
        show_progress=True
    )
    
    if results:
        print(f"\nProcessing completed!")
        print(f"   Output video: {output_video_path}")
else:
    print("Unable to process video - file not found")
    results = None

Loading model: c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\results\weights\best.pt
Video found: demo_video.mp4

Processing video...


Analysis in progress: 100%|██████████| 1815/1815 [01:00<00:00, 29.92it/s]



Processing completed!
   Output video: c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\results\traffic_analysis\analyzed_demo_video.mp4
